# ADRI Guard Decorator Tutorial

This tutorial demonstrates how to use the `adri_guarded` decorator to enforce data quality standards in your agent functions.

The `adri_guarded` decorator assesses the quality of a data source before allowing the decorated function to proceed. If the data quality doesn't meet the minimum score it raises a ValueError with details about the issues.

## Setup

First let's make sure we have ADRI installed:

In [ ]:
# Install ADRI if not already installed
!pip install -e ".." --quiet

Now let's import the necessary modules:

In [ ]:
import os
import pandas as pd
from adri.integrations import adri_guarded

## Creating a Sample Dataset

Let's create a couple of sample datasets to demonstrate the decorator - one with good quality and one with poor quality:

In [ ]:
# Create a high-quality dataset
high_quality_data = pd.DataFrame({
    'customer_id': [1001, 1002, 1003, 1004, 1005],
    'name': ['John Smith', 'Jane Doe', 'Bob Johnson', 'Alice Brown', 'Charlie Davis'],
    'email': ['john@example.com', 'jane@example.com', 'bob@example.com', 'alice@example.com', 'charlie@example.com'],
    'age': [32, 28, 45, 36, 51],
    'purchase_amount': [125.50, 89.99, 210.75, 55.25, 175.00]
})

# Create a low-quality dataset with missing values and inconsistencies
low_quality_data = pd.DataFrame({
    'customer_id': [2001, 2002, None, 2004, 2005],
    'name': ['John Smith', None, 'Bob Johnson', 'Alice Brown', None],
    'email': ['not-an-email', 'jane@example.com', None, 'alice@example', 'charlie'],
    'age': [132, -28, 45, None, 51],  # Age 132 is implausible, -28 is invalid
    'purchase_amount': [None, 89.99, 210.75, 55.25, -175.00]  # Negative amount is invalid
})

# Save the datasets to CSV files
high_quality_data.to_csv('high_quality_data.csv', index=False)
low_quality_data.to_csv('low_quality_data.csv', index=False)

print("High-quality dataset:")
display(high_quality_data)

print("\nLow-quality dataset:")
display(low_quality_data)

## Creating a Function with the ADRI Guard Decorator

Now let's create a function that analyzes customer data and decorate it with `adri_guarded`:

In [ ]:
@adri_guarded(min_score=70)
def analyze_customer_data(data_source, analysis_type='segmentation'):
    """
    Analyze customer data for insights.
    
    This function will only run if data_source meets the minimum quality score of 70.
    
    Args:
        data_source: Path to the data source
        analysis_type: Type of analysis to perform
        
    Returns:
        dict: Analysis results
    """
    print(f"Analyzing {data_source} for {analysis_type}...")
    
    # In a real application, this would perform actual analysis
    # For this tutorial, we'll just read the data and return some basic statistics
    df = pd.read_csv(data_source)
    
    results = {
        'record_count': len(df),
        'field_count': len(df.columns),
        'numeric_fields': {
            col: {
                'mean': df[col].mean(),
                'min': df[col].min(),
                'max': df[col].max()
            }
            for col in df.select_dtypes(include=['number']).columns
        }
    }
    
    print(f"Analysis of {data_source} complete!")
    return results

## Testing with High-Quality Data

Let's try running our function with the high-quality dataset:

In [ ]:
try:
    results = analyze_customer_data('high_quality_data.csv')
    print("\nAnalysis results:")
    for key, value in results.items():
        print(f"{key}: {value}")
except ValueError as e:
    print(f"Analysis blocked: {e}")

## Testing with Low-Quality Data

Now let's try running our function with the low-quality dataset:

In [ ]:
try:
    results = analyze_customer_data('low_quality_data.csv')
    print("\nAnalysis results:")
    for key, value in results.items():
        print(f"{key}: {value}")
except ValueError as e:
    print(f"Analysis blocked: {e}")

## Customizing the Decorator

The `adri_guarded` decorator can be customized with different parameters:

In [ ]:
@adri_guarded(min_score=30, data_source_param='file_path')
def generate_report(file_path, report_type='summary'):
    """
    Generate a report from data.
    
    This function uses a different parameter name for the data source
    and has a lower quality threshold.
    
    Args:
        file_path: Path to the data file
        report_type: Type of report to generate
        
    Returns:
        str: Report content
    """
    print(f"Generating {report_type} report from {file_path}...")
    
    # In a real application, this would generate an actual report
    df = pd.read_csv(file_path)
    report = f"Report for {file_path}\n"
    report += f"Number of records: {len(df)}\n"
    report += f"Columns: {' '.join(df.columns)}\n"
    
    print(f"Report generation complete!")
    return report

In [ ]:
try:
    report = generate_report('low_quality_data.csv')
    print("\nGenerated report:")
    print(report)
except ValueError as e:
    print(f"Report generation blocked: {e}")

## Conclusion

The `adri_guarded` decorator provides a simple way to enforce data quality standards in your agent functions. By using this decorator, you can ensure that your agents only work with data that meets your quality requirements.

Key benefits:

1. **Automatic Quality Checks**: The decorator automatically assesses data quality before executing the function
2. **Customizable Thresholds**: You can set different minimum quality scores for different functions
3. **Detailed Error Messages**: When data quality is insufficient, the decorator provides detailed information about the issues
4. **Flexible Parameter Names**: The decorator can work with any parameter name for the data source

For more information, see the [INTEGRATIONS.md](../INTEGRATIONS.md) guide.

## Cleanup

Let's clean up the sample files we created:

In [ ]:
# Remove the sample files
os.remove('high_quality_data.csv')
os.remove('low_quality_data.csv')
print("Sample files removed.")